### Imports

In [16]:
%load_ext autoreload
%autoreload 2

# Internal import
import deep_learning_land_use_classification.config as config
from deep_learning_land_use_classification.dataset import get_single_label_data
import deep_learning_land_use_classification.evaluation as evaluation

# External imports
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Get data
train_df, test_df, class_names, num_classes = get_single_label_data()

In [4]:
print(train_df.head())

                                                   path       label
1592  C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...  parkinglot
829   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...     freeway
414   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...   buildings
704   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...      forest
755   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...      forest


In [5]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="sstaheli52-wageningen-university-and-research",
    
    # Set the wandb project where this run will be logged.
    project="DL_single-label-land-use-classification",
    dir = config.WANDB_DIR,

    # Track hyperparameters and run metadata.
    config={
        "architecture": "resnet50",
        "pretrained": True,
        "pretraining_dataset": "ImageNetV2",
        "pretraining_source": "torchvision",
        "weights": "IMAGENET1K_V2",
        "epochs": 5,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "optimizer": "Adam",
        "loss": "BCEWithLogitsLoss",
        "num_classes": num_classes
        },
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tomer\_netrc.
wandb: Currently logged in as: tomer-peled (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Resize, transform and normalize dataset

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # standard ImageNet mean
        std =[0.229, 0.224, 0.225] # standard ImageNet std
    )
])

### Get training and test dataset, as well as dataset loaders

In [26]:
class SingleLabelDataset(Dataset):
    def __init__(self, df, class_names, transform=None):
        self.df = df.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform
        self.label_to_idx = {label: i for i, label in enumerate(class_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        label = self.label_to_idx[row["label"]]
        return image, torch.tensor(label, dtype=torch.long)

train_dataset = SingleLabelDataset(train_df, class_names, transform)
test_dataset  = SingleLabelDataset(test_df, class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=run.config.batch_size, shuffle=False)

### Initiate model

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Using device: cuda


In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [29]:
run.watch(model, log="all", log_freq=100)

### Train model

In [30]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [31]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
epochs = run.config.epochs

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")
    run.log({"train_loss": train_loss, "test_loss": test_loss})
    
    # Log metrics to Weights & Biases
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_singlelabel(
        model,
        test_loader,
        device
    )
    class_metrics = wandb.Table(columns=["class", "precision", "recall", "f1"])
    for i, class_name in enumerate(class_names):
        class_metrics.add_data(class_name, float(precision[i]), float(recall[i]), float(f1[i]))
        print(f"{class_name:15s} | Precision: {precision[i]:.3f} | Recall: {recall[i]:.3f} | F1: {f1[i]:.3f}")

    print ("macro precision:", p_macro)
    print ("macro recall:", r_macro)
    print ("macro f1:", f1_macro)
    
    run.log({
        "train_loss": train_loss,
        "test_loss": test_loss,
        "macro/precision": p_macro,
        "macro/recall": r_macro,
        "macro/f1": f1_macro,
        "class_metrics": class_metrics,
    })
    
    print("")
    print("--------------------------------")
    print ("")
    
run.finish()